# 🚦 L06　分類與評估
**統計冒險之旅 2026**　｜　Day 2（09/24 四）⛰️ 模型之嶺　｜　關卡　｜　🏅 100 XP

📖 ISLP Ch4–5 觀念；資料：固定種子合成 Default-like 信用資料

### 🎯 這一關你會學到
- LogisticRegression、predict_proba 與門檻
- 混淆矩陣、精確率、召回率、F1
- 用 train/dev 選門檻與模型，final test 只報告一次
- ROC 曲線與 AUC；KNN 分類器比較

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
> 🎲 切分、抽樣、模型請照題目使用 `random_state=42`。開發時只看 dev；final test 到最後一題才揭曉，而且看完不回頭調整。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L06"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["6-1", "6-2", "6-3", "6-4", "6-5", "6-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_6_1(run):
    out, ns = run()
    for name, rows in [("X_train", 6000), ("X_dev", 2000), ("X_test", 2000)]:
        ok, msg = 資料框像(抓變數(ns, name), 列=rows, 欄=3)
        if not ok: return (False, f"{name}：{msg}")
    if [int(抓變數(ns, n).sum()) for n in ["y_train", "y_dev", "y_test"]] != [197, 65, 66]:
        return (False, "兩次切分都要使用 stratify 與 random_state=42。")
    return (約等於(抓變數(ns, "違約率"), 0.03280, 0.0005), "違約率 = y.mean()。")
任務定義("6-1", _check_6_1, 提示="先保留 20% final test，再把其餘資料切出 25% 作 dev；兩次都 stratify。")

def _check_6_2(run):
    if "X_test" in run.src or "y_test" in run.src:
        return (False, "調整模型時不能讀 final test；這題只使用 X_train、y_train 與 X_dev、y_dev。")
    out, ns = run()
    if not 約等於(抓變數(ns, "dev平均機率"), 0.03408, 0.002): return (False, "dev機率 = clf.predict_proba(X_dev)[:, 1]。")
    if not 約等於(抓變數(ns, "dev準確率"), 0.97400, 0.002): return (False, "dev準確率 = (dev預測 == y_dev).mean()。")
    return (約等於(抓變數(ns, "dev全猜不違約準確率"), 0.96750, 0.002), "基線 = 1 - y_dev.mean()。")
任務定義("6-2", _check_6_2, 提示="predict_proba 的第 1 欄是違約機率；開發階段只看 dev。")

def _check_6_3(run):
    if "X_test" in run.src or "y_test" in run.src:
        return (False, "門檻分析只能使用 dev，final test 要留到最後一題。")
    out, ns = run()
    if not 約等於(抓變數(ns, "dev召回率"), 0.30769, 0.005): return (False, "dev召回率 = recall_score(y_dev, dev預測)。")
    if not 約等於(抓變數(ns, "dev精確率"), 0.74074, 0.005): return (False, "dev精確率 = precision_score(y_dev, dev預測)。")
    return (int(抓變數(ns, "dev漏掉人數")) == 45, "漏掉人數 = dev混淆矩陣[1, 0]。")
任務定義("6-3", _check_6_3, 提示="confusion_matrix 的排列是 [[TN, FP], [FN, TP]]；這題用 y_dev。")

def _check_6_4(run):
    if "X_test" in run.src or "y_test" in run.src:
        return (False, "門檻要在 dev 選，不能先看 final test。")
    out, ns = run()
    if not 約等於(抓變數(ns, "dev召回率02"), 0.63077, 0.005): return (False, "dev 門檻 0.2：(dev機率 >= 0.2)。")
    if not 約等於(抓變數(ns, "dev精確率02"), 0.42268, 0.005): return (False, "dev精確率02 不對。")
    if not 約等於(抓變數(ns, "建議門檻"), 0.2, 1e-9): return (False, "本題依 dev 比較後把建議門檻設為 0.2。")
    return (bool(抓變數(ns, "召回率變高")) and bool(抓變數(ns, "精確率變低")), "門檻降低後檢查 dev 的 recall／precision 取捨。")
任務定義("6-4", _check_6_4, 提示="只改 dev 的決策門檻；final test 留到 6-6。")

def _check_6_5(run):
    if "X_test" in run.src or "y_test" in run.src:
        return (False, "開發 AUC 與 ROC 請使用 dev；final test 不用來畫調整中的圖。")
    out, ns = run()
    if not 約等於(抓變數(ns, "devAUC"), 0.94009, 0.003): return (False, "devAUC = roc_auc_score(y_dev, dev機率)。")
    if not run.figs: return (False, "沒有畫出圖。")
    f = run.figs[0]
    return ("ROC" in f["title"].upper() and f["n_lines"] >= 2 and f["legend"], "標題含 ROC、兩條線、有圖例。")
任務定義("6-5", _check_6_5, 提示="roc_auc_score 的第二個參數是 dev機率。")

def _check_6_6(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "devAUC_KNN_未標準化"), 0.87796, 0.005): return (False, "KNN 未標準化要在 dev 比較。")
    if not 約等於(抓變數(ns, "devAUC_KNN_標準化"), 0.87028, 0.005): return (False, "KNN 標準化也要在同一份 dev 比較。")
    if str(抓變數(ns, "最佳模型")) != "邏輯斯": return (False, "只依 devAUC 選模型；這次應選到邏輯斯。")
    if not 約等於(抓變數(ns, "最終測試AUC"), 0.95303, 0.003): return (False, "選定模型後才用 final test 計算一次最終測試AUC。")
    if not 約等於(抓變數(ns, "最終測試召回率"), 0.59091, 0.005): return (False, "final test 只套用 dev 選好的 0.2 門檻。")
    return (約等於(抓變數(ns, "最終測試精確率"), 0.37500, 0.005), "final test 只做一次不回頭調整的報告。")
任務定義("6-6", _check_6_6, 提示="先用 devAUC 選模型，再把選定模型與建議門檻原封不動套到 final test。")

## 🚦 6-1　從預測數字到預測類別
「會不會違約？」「會不會回購？」答案是**類別**，不是數字。直接用線性迴歸會算出 −0.3 或 1.4 這種「機率破表」的數字。
**邏輯斯迴歸（logistic regression）**把直線壓進 0～1 之間（S 型曲線），輸出的是**機率**——例如「這位客戶違約的機率 0.08」。
資料：以固定種子產生的 10,000 筆 Default-like 合成信用紀錄，欄位沿用 default、student、balance、income 結構；不含真實客戶資料，也不是 ISLP 原始資料。

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, roc_auc_score, roc_curve
df = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0/data/default.csv")
print(df["default"].value_counts(normalize=True).round(4))     # 約 3.3% 違約 → 類別不平衡
df.head()

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
y = (df["default"] == "Yes").astype(int)
X = pd.DataFrame({"balance": df["balance"], "income": df["income"], "student": (df["student"] == "Yes").astype(int)})
X_work, X_test, y_work, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_dev, y_train, y_dev = train_test_split(X_work, y_work, test_size=0.25, random_state=42, stratify=y_work)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
dev機率 = clf.predict_proba(X_dev)[:, 1]
print("train/dev/final test", X_train.shape, X_dev.shape, X_test.shape)
print("dev 前 5 位違約機率", dev機率[:5].round(3))
grid = pd.DataFrame({"balance": np.linspace(0, 2700, 200), "income": X_train["income"].mean(), "student": 0})
plt.plot(grid["balance"], clf.predict_proba(grid)[:, 1]); plt.xlabel("balance"); plt.ylabel("違約機率"); plt.title("S 型曲線"); plt.show()

## 6-2　門檻與混淆矩陣：健檢報告的四格
機率要變成「會／不會」需要一條**門檻**（預設 0.5）。對照真實答案，會得到四格：

| | 預測：違約 | 預測：不違約 |
|---|---|---|
| **真的違約** | 真陽 TP（抓到） | 假陰 FN（漏掉！） |
| **真的沒違約** | 假陽 FP（誤判） | 真陰 TN |

- **準確率** = 猜對的比例。⚠️ 類別不平衡時是**虛榮指標**：全部猜「不違約」也有 96.7%。
- **召回率 recall** = 真的違約裡抓到幾成（漏掉的代價高時看這個）。
- **精確率 precision** = 被判違約的裡面有幾成是真的（誤判的代價高時看這個）。
- **F1** = 兩者的調和平均。

In [ ]:
dev預測 = (dev機率 >= 0.5).astype(int)
print("dev 準確率", round((dev預測 == y_dev).mean(), 4), "| dev 全猜不違約", round(1 - y_dev.mean(), 4))
print(confusion_matrix(y_dev, dev預測))
print(classification_report(y_dev, dev預測, digits=3))

## 6-3　門檻往下調：入場年齡線放寬
銀行寧可多查幾個人，也不想漏掉真的會違約的人 → 在 **dev** 把門檻從 0.5 降到 0.2，比較召回率與精確率。門檻不是由 final test 挑選；final test 只在選定規則後評估一次。

In [ ]:
for t in [0.5, 0.3, 0.2, 0.1]:
    p = (dev機率 >= t).astype(int)
    print(f"dev 門檻 {t}：召回率 {recall_score(y_dev, p):.3f}　精確率 {precision_score(y_dev, p):.3f}　抓出 {p.sum()} 人")

## 6-4　ROC 曲線與 AUC：把門檻從最嚴掃到最鬆
每個門檻都有一組（假陽率, 真陽率），把所有門檻連起來就是 **ROC 曲線**；曲線下的面積 **AUC**（0.5 = 亂猜，1 = 完美）＝「隨便抓一個違約的和一個沒違約的，模型把違約的排前面的機率」。AUC 不看門檻，是比較模型最常用的分數。

In [ ]:
fpr, tpr, thr = roc_curve(y_dev, dev機率)
plt.plot(fpr, tpr, label=f"邏輯斯迴歸 dev AUC = {roc_auc_score(y_dev, dev機率):.3f}"); plt.plot([0, 1], [0, 1], "--", color="grey", label="亂猜")
plt.xlabel("假陽率"); plt.ylabel("真陽率（召回率）"); plt.title("ROC 曲線（dev）"); plt.legend(); plt.show()

## 6-5　KNN 分類器與標準化
KNN 也能分類（看鄰居多數決）。但 KNN 靠「距離」，income（幾萬）會壓過 balance（幾百）——不標準化，模型幾乎只看 income。**Pipeline 裡先放 StandardScaler**，同一把尺再量距離。

In [ ]:
for name, m in [("KNN 未標準化", KNeighborsClassifier(15)), ("KNN 標準化", make_pipeline(StandardScaler(), KNeighborsClassifier(15)))]:
    m.fit(X_train, y_train)
    print(f"{name}：dev AUC {roc_auc_score(y_dev, m.predict_proba(X_dev)[:, 1]):.3f}")

### 🎯 任務 6-1　建立目標與 train/dev/final test

建立 `y` 與三欄 `X`。先以 `test_size=0.2, random_state=42, stratify=y` 保留 `X_test/y_test`；再把 `X_work/y_work` 以 `test_size=0.25, random_state=42, stratify=y_work` 切成 `train` 與 `dev`。最後算 `違約率`。

In [ ]:
# 🎯 任務 6-1　建立目標與切分（請保留這一行）
y = (df["default"] == "Yes").astype(int)
X = pd.DataFrame({"balance": df["balance"], "income": df["income"], "student": (df["student"] == "Yes").astype(int)})
X_work, X_test, y_work, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=???)
X_train, X_dev, y_train, y_dev = train_test_split(X_work, y_work, test_size=0.25, random_state=42, stratify=???)
違約率 = ???
print(X_train.shape, X_dev.shape, X_test.shape, round(違約率, 4))

In [ ]:
檢查("6-1")   # ◀ 執行這一格，看看任務 6-1 有沒有過關

### 🎯 任務 6-2　邏輯斯迴歸與 dev 機率

用 Pipeline（StandardScaler → LogisticRegression）只在 train 建立 `clf`，算 `dev機率`、`dev平均機率`；以 0.5 得到 `dev預測`，再算 `dev準確率` 與 `dev全猜不違約準確率`。這一題不可讀取 final test。

In [ ]:
# 🎯 任務 6-2　邏輯斯迴歸與機率（請保留這一行）
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
dev機率 = clf.predict_proba(X_dev)[:, ???]
dev平均機率 = dev機率.mean()
dev預測 = (dev機率 >= 0.5).astype(int)
dev準確率 = (dev預測 == y_dev).mean()
dev全猜不違約準確率 = ???
print(round(dev平均機率, 4), round(dev準確率, 4), round(dev全猜不違約準確率, 4))

In [ ]:
檢查("6-2")   # ◀ 執行這一格，看看任務 6-2 有沒有過關

### 🎯 任務 6-3　dev 混淆矩陣、召回率、精確率

用 `y_dev` 與 `dev預測` 算 `dev混淆矩陣`、`dev召回率`、`dev精確率`，並把假陰 FN 存成 `dev漏掉人數`。final test 仍保持封存。

In [ ]:
# 🎯 任務 6-3　混淆矩陣、召回率、精確率（請保留這一行）
dev混淆矩陣 = confusion_matrix(y_dev, dev預測)
dev召回率 = recall_score(y_dev, dev預測)
dev精確率 = ???
dev漏掉人數 = dev混淆矩陣[???, ???]
print(dev混淆矩陣); print(round(dev召回率, 3), round(dev精確率, 3), dev漏掉人數)

In [ ]:
檢查("6-3")   # ◀ 執行這一格，看看任務 6-3 有沒有過關

### 🎯 任務 6-4　在 dev 比較 0.5 與 0.2

只用 `dev機率` 把門檻降到 0.2，算 `dev召回率02`、`dev精確率02`，比較兩項指標並將 `建議門檻` 設為 0.2。這個門檻選定後，不能因 final test 結果再改。

In [ ]:
# 🎯 任務 6-4　把門檻降到 0.2（請保留這一行）
dev預測02 = (dev機率 >= ???).astype(int)
dev召回率02 = recall_score(y_dev, dev預測02)
dev精確率02 = precision_score(y_dev, dev預測02)
召回率變高 = dev召回率02 > dev召回率
精確率變低 = ???
建議門檻 = ???
print(round(dev召回率02, 3), round(dev精確率02, 3), 召回率變高, 精確率變低)

In [ ]:
檢查("6-4")   # ◀ 執行這一格，看看任務 6-4 有沒有過關

### 🎯 任務 6-5　dev ROC 與 AUC

算 `devAUC = roc_auc_score(y_dev, dev機率)`，並只用 dev 畫 ROC 曲線、亂猜對角線與圖例。

In [ ]:
# 🎯 任務 6-5　ROC 與 AUC（請保留這一行）
devAUC = ???
fpr, tpr, thr = roc_curve(y_dev, dev機率)
plt.plot(fpr, tpr, label=f"dev AUC = {devAUC:.3f}"); plt.plot([0, 1], [0, 1], "--", color="grey", label="亂猜")
plt.xlabel("假陽率"); plt.ylabel("真陽率"); plt.title(???); plt.legend(); plt.show()
print(round(devAUC, 4))

In [ ]:
檢查("6-5")   # ◀ 執行這一格，看看任務 6-5 有沒有過關

### 🎯 任務 6-6　用 dev 選模型，final test 只評一次

K=15 比較 Logistic、KNN 未標準化與標準化的 **dev AUC**，存成 `最佳模型`。選定後才以該模型對 `X_test` 產生一次 `最終機率`，套用先前的 `建議門檻`，報告 `最終測試AUC`、`最終測試召回率`、`最終測試精確率`；不依測試結果回頭改模型或門檻。

In [ ]:
# 🎯 任務 6-6　KNN 與標準化（請保留這一行）
knn_raw = KNeighborsClassifier(15).fit(X_train, y_train)
knn_std = make_pipeline(StandardScaler(), KNeighborsClassifier(15)).fit(X_train, y_train)
devAUC_KNN_未標準化 = roc_auc_score(y_dev, knn_raw.predict_proba(X_dev)[:, 1])
devAUC_KNN_標準化 = ???
dev分數 = {"邏輯斯": devAUC, "KNN標準化": devAUC_KNN_標準化, "KNN未標準化": devAUC_KNN_未標準化}
最佳模型 = max(dev分數, key=dev分數.get)
最終模型 = {"邏輯斯": clf, "KNN標準化": knn_std, "KNN未標準化": knn_raw}[最佳模型]
最終機率 = 最終模型.predict_proba(X_test)[:, 1]
最終測試AUC = roc_auc_score(y_test, 最終機率)
最終預測 = (最終機率 >= 建議門檻).astype(int)
最終測試召回率 = recall_score(y_test, 最終預測)
最終測試精確率 = precision_score(y_test, 最終預測)
print({k: round(v, 3) for k, v in dev分數.items()}, 最佳模型)
print("final test", round(最終測試AUC, 3), round(最終測試召回率, 3), round(最終測試精確率, 3))

In [ ]:
檢查("6-6")   # ◀ 執行這一格，看看任務 6-6 有沒有過關

## 🌟 進階挑戰（不計分）
1. 只用 `balance` 一欄，在 train 建模並用 dev 比較 AUC；不要查看 final test。
2. 只用 dev 找出「召回率 ≥ 0.8」的最高門檻，再將規則鎖定。

---
## 🔑 通關密語
　你已經會把機率變成決策，也知道準確率為什麼會騙人。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：⚔️ B2 Boss 戰：心臟病風險預警** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.0.0/notebooks/B2_boss_heart.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/